# InterviewIQ / CSS Mock Interview Coach — Knowledge Base Builder (Colab)

**Terminology:** throughout this notebook, "CSS" means the **Central Superior Services Examination**
conducted by Pakistan's Federal Public Service Commission (FPSC). It never means Cascading Style
Sheets or anything web-development related.

**Standalone research notebook.** This runs entirely on Google Colab's cloud compute and Google Drive
storage — it does not run on your laptop and is independent of the `css-mock-interview-coach` backend
and its hardware rules (no local models / no large embedding models / no GPU on your machine).

**Source catalog:** `research/colab/sources.json` (next to this notebook in the repo) lists every
source below plus how each one was verified (HTTP status, actual extracted text, not just a search
snippet) and a list of sources that were checked and rejected (dead links, scanned/non-text PDFs,
JS-rendered pages that don't scrape), so you can see what's already been tried before adding more.

**What this notebook does**

Collects CSS-relevant source material (PDFs, web articles, YouTube transcripts), cleans it, chunks it,
embeds it with a small sentence-transformer model, and builds a FAISS vector index — a general CSS
*knowledge base* an interviewer/RAG agent could search for subject background.

**What this notebook is NOT**

It does not produce the structured interview *question bank* the app actually loads
(`data/questions/*.json`, validated by `backend/app/rag/dataset.py`). For that, use
`backend/scripts/generate_questions.py` in the main repo, which drafts questions in the exact schema
the app expects and never writes into the live dataset without human review.

**Before wiring this into the backend:** the running app currently does keyword retrieval only
(`backend/app/rag/retriever.py`), by design, to stay within the project's "no large embedding models"
hardware rule. Adding FAISS + sentence-transformers to the FastAPI backend itself would be a real
architecture change needing explicit sign-off — this notebook only prepares data; it doesn't make that
call for you.

**Licensing / scraping reminder:** only use PDFs/books you can legally access, respect site terms and
`robots.txt` when scraping, and only pull YouTube transcripts for videos where that's permitted. Always
keep source title, URL/date and page number with every chunk — never present scraped or AI-processed
text as an official CSS/FPSC source.

## 1. Install dependencies (Colab runtime only)

In [1]:
!pip install -q pymupdf trafilatura youtube-transcript-api sentence-transformers faiss-cpu requests truststore



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: C:\Users\kinza\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


## 2. Storage setup: Google Drive (Colab) or a local folder (VS Code / any local Jupyter kernel)

Creates `InterviewIQ_Dataset/` with the 10 subfolders used throughout this notebook. Safe to re-run.

**Environment-aware, not Colab-only:** this cell detects whether `google.colab` is importable. In real
Colab it mounts Drive exactly as before -- nothing about the Colab path changed. Outside Colab (a local
Jupyter kernel in VS Code, for example) there is no Drive to mount, so it falls back to a plain local
folder, `research/colab/output/InterviewIQ_Dataset/` -- **this is ordinary local disk, not Google
Drive**; nothing here is synced, backed up, or shared the way a Drive folder would be, and the two
storage locations are never silently conflated. If you need the output to land on Drive from a local
machine, mount Drive yourself (e.g. with `rclone` or the Drive desktop client) and point
`LOCAL_OUTPUT_DIR` below at that mount -- this notebook does not do that for you.

In [2]:
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/InterviewIQ_Dataset")
else:
    import os
    # CSS_RAG_LOCAL_OUTPUT_DIR lets a specific run (e.g. validating a pipeline change into a clearly
    # separate, clearly named directory) override the default without changing this cell's behavior
    # for ordinary runs. Unset -> same "output/InterviewIQ_Dataset" default as always.
    LOCAL_OUTPUT_DIR = Path(os.environ.get("CSS_RAG_LOCAL_OUTPUT_DIR", "output/InterviewIQ_Dataset")).resolve()
    BASE_DIR = LOCAL_OUTPUT_DIR
    print(f"Not running in Google Colab -- using a local folder, NOT Google Drive: {BASE_DIR}")

SUBFOLDERS = [
    "01_Raw_Data", "02_Web_Articles", "03_YouTube_Transcripts", "04_Candidate_Profiles",
    "05_Interview_Question_Bank", "06_Cleaned_Data", "07_Processed_Chunks",
    "08_Embeddings", "09_Evaluation_Dataset", "10_Final_Export",
]
for name in SUBFOLDERS:
    (BASE_DIR / name).mkdir(parents=True, exist_ok=True)

print("Folders ready under:", BASE_DIR)


Not running in Google Colab -- using a local folder, NOT Google Drive: C:\Users\kinza\OneDrive\Desktop\css-mock-interview-coach\research\colab\output_promoted_baseline\InterviewIQ_Dataset
Folders ready under: C:\Users\kinza\OneDrive\Desktop\css-mock-interview-coach\research\colab\output_promoted_baseline\InterviewIQ_Dataset


## 3. Configuration

Edit the subject list / chunking settings here. `SUBJECT_LABELS` is informational metadata on chunks —
it does not have to match the backend's `SUBJECTS` constant exactly, since this knowledge base is for
background retrieval, not the structured question bank.

In [3]:
SUBJECT_LABELS = [
    "Pakistan Affairs", "Current Affairs", "Islamic Studies", "General Knowledge",
    "English", "International Relations", "Governance and Constitutional Studies",
    "Optional: International Relations", "Optional: Political Science", "Optional: Public Administration",
]

CHUNK_SIZE_WORDS = 350      # approx. 500 tokens
CHUNK_OVERLAP_WORDS = 70    # approx. 100 tokens
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K_DEFAULT = 5


## 4. Extraction functions

One function per source type. Each returns plain text plus the metadata needed later — never bare text
with no provenance.

`with_retries` wraps any of the fetch calls below with retries + backoff, and `PROCESSING_LOG` tracks
which sources succeeded or failed so section 9b can print a full success/failure report (per the
reliability requirements in this notebook's task brief: handle inaccessible URLs gracefully, retry, log,
and track outcomes).

In [4]:
import time

PROCESSING_LOG = {"success": [], "failed": []}


def with_retries(fn, *args, retries: int = 3, backoff_seconds: float = 3.0, **kwargs):
    """Calls fn(*args, **kwargs), retrying on exception with linear backoff. Re-raises the last error."""
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            last_exc = exc
            print(f"    attempt {attempt}/{retries} failed: {exc}")
            if attempt < retries:
                time.sleep(backoff_seconds * attempt)
    raise last_exc


**Why `truststore` here, confirmed by diagnosis, not guessed:** `fpsc.gov.pk` and `na.gov.pk` failed
with `SSLCertVerificationError: unable to get local issuer certificate` under plain `requests`/`certifi`,
even though the same URLs open fine in a browser or `curl`. Testing showed `curl` succeeds with the
*exact same* CA bundle `requests` uses, which rules out an outdated `certifi` bundle -- the real cause is
that those servers don't send their intermediate certificate in the TLS handshake, and the OS trust
store (which curl/browsers consult) fetches it automatically (AIA chasing) while Python's static
`certifi` bundle does not. `truststore.inject_into_ssl()` makes `ssl`/`requests` use the OS trust store
instead, which performs that same AIA chasing -- **certificate verification stays fully enabled**, this
only changes which trust store does the verifying. Confirmed locally: all three previously-failing PDFs
return HTTP 200 with this change, no other change needed.

In [5]:
import truststore
truststore.inject_into_ssl()  # use the OS trust store (fixes incomplete-chain govt sites); never disables verification

import requests


def download_file(url: str, dest_path: Path, user_agent: str = "Mozilla/5.0 (compatible; InterviewIQ-notebook)") -> None:
    """Downloads url to dest_path, retrying transient failures. Raises on a non-2xx status after retries."""
    def _get():
        resp = requests.get(url, headers={"User-Agent": user_agent}, timeout=30)
        resp.raise_for_status()
        dest_path.write_bytes(resp.content)

    with_retries(_get, retries=3, backoff_seconds=3.0)


def download_or_explain(url: str, dest_path: Path, source_title: str) -> None:
    """download_file, but on failure raises with exact manual-fallback instructions instead of a bare
    traceback -- certificate/network issues on government sites shouldn't silently lose a source."""
    try:
        download_file(url, dest_path)
    except Exception as exc:
        raise RuntimeError(
            f"Automatic download failed for '{source_title}' ({exc}). "
            f"Manual fallback: open {url} in your browser, save the PDF, then upload it to "
            f"Google Drive at InterviewIQ_Dataset/01_Raw_Data/{dest_path.name} and re-run this cell -- "
            f"download_or_explain only downloads when the local file doesn't already exist."
        ) from exc


In [6]:
import re
import fitz  # PyMuPDF


def extract_pdf_text(pdf_path: Path) -> list[dict]:
    """Returns one record per page: {"page_number": int, "text": str}."""
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc, start=1):
        text = page.get_text("text")
        if text.strip():
            pages.append({"page_number": i, "text": text})
    doc.close()
    return pages


C:\Users\kinza\OneDrive\Desktop\css-mock-interview-coach\.venv-rag\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [7]:
import trafilatura


def extract_web_article(url: str) -> dict | None:
    """Returns {"title": str, "text": str, "url": str} or None if extraction failed."""
    downloaded = trafilatura.fetch_url(url)
    if not downloaded:
        return None
    text = trafilatura.extract(downloaded, include_comments=False, include_tables=False)
    if not text:
        return None
    metadata = trafilatura.extract_metadata(downloaded)
    title = metadata.title if metadata and metadata.title else url
    return {"title": title, "text": text, "url": url}


In [8]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound


def fetch_youtube_transcript(video_id: str) -> str | None:
    """Returns the transcript text, or None if no captions are available for this video."""
    try:
        api = YouTubeTranscriptApi()
        transcript = api.fetch(video_id)
        return " ".join(snippet.text for snippet in transcript)
    except (TranscriptsDisabled, NoTranscriptFound):
        return None


## 5. Cleaning and chunking

In [9]:
def clean_text(text: str) -> str:
    text = text.replace("\r", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE_WORDS, overlap: int = CHUNK_OVERLAP_WORDS) -> list[str]:
    words = text.split()
    if not words:
        return []
    chunks, start = [], 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return chunks


## 6. Build chunk records with metadata

One record per chunk, matching the metadata shape: `document_id`, `subject`, `topic`, `source_type`,
`source_title`, `source_url`, `page_number`, `language`, `content`, `chunk_id`.

**Stable IDs, fixed:** an earlier version derived `document_id` from `uuid.uuid4()` -- a fresh random id
every call, so re-running a section (very normal while iterating in a notebook) silently produced a
second, differently-ID'd copy of every chunk instead of recognizing it as the same content. `document_id`
is now a deterministic hash of the source (same source -> same id, every run), and `chunk_id` additionally
hashes in the page number and position so it's stable too. This makes de-duplication by `chunk_id` or
`document_id` meaningful, and is required before trusting any later "N distinct chunks" count across
multiple runs.

In [10]:
import hashlib
import json


def _stable_id(*parts: str, length: int = 12) -> str:
    return hashlib.sha1("|".join(str(p) for p in parts).encode("utf-8")).hexdigest()[:length]


def make_chunk_records(
    text: str,
    *,
    subject: str,
    topic: str,
    source_type: str,
    source_title: str,
    source_url: str | None,
    page_number: int | None = None,
    language: str = "English",
) -> list[dict]:
    document_id = _stable_id(source_url or source_title)
    records = []
    for i, chunk in enumerate(chunk_text(clean_text(text)), start=1):
        records.append({
            "document_id": document_id,
            "subject": subject,
            "topic": topic,
            "source_type": source_type,       # "book" | "official_document" | "news_article" | "youtube_transcript" | "educational_lecture"
            "source_title": source_title,
            "source_url": source_url,
            "page_number": page_number,
            "language": language,
            "content": chunk,
            "chunk_id": _stable_id(source_url or source_title, page_number, i, length=16),
        })
    return records


def append_jsonl(records: list[dict], out_path: Path) -> None:
    with open(out_path, "a", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


## 7. Process official CSS/Pakistan PDFs → chunks

These are real, individually-verified official sources (checked live: HTTP status plus actual extracted
text, not just a search-result snippet — see `research/colab/sources.json` for how and when). Each
entry is downloaded straight from its official URL into `01_Raw_Data/`, so there's nothing to upload by
hand for this starter set. Add more entries the same way once you've verified them yourself.

FPSC's own site restructures its document paths every exam cycle, so re-verify the two `fpsc.gov.pk`
URLs below before relying on them in a future run — they were live as of 2026-11-17.

**SSL note (already handled above, confirmed working):** `fpsc.gov.pk` and `na.gov.pk` used to fail under
plain `requests` with `SSLCertVerificationError: unable to get local issuer certificate`, because those
servers omit their intermediate certificate and Python's static `certifi` bundle can't chase it down
(browsers/curl can). Section 4's `truststore.inject_into_ssl()` fixes this by using the OS trust store
instead -- confirmed locally: all three previously-failing URLs now return HTTP 200, **with certificate
verification still fully enabled**. If a download still fails for some other reason (the URL rotated,
the site is down), `download_or_explain` below raises with the exact manual steps: open the URL in a
browser, save the PDF, and drop it into `01_Raw_Data/<filename>` yourself -- it only downloads when the
local file doesn't already exist, so a manually-placed file is picked up automatically next run.

In [11]:
PDF_JOBS = [
    {
        "url": "https://www.fpsc.gov.pk/uploads/content/1794127244067_CSS_2027_Advertisement.pdf",
        "filename": "fpsc_css_ce_2027_public_notice.pdf",
        "subject": "General Knowledge", "topic": "CSS Exam Structure and Eligibility",
        "source_title": "CSS Competitive Examination (CE-2027) Public Notice",
    },
    {
        "url": "https://www.fpsc.gov.pk/uploads/content/1785753970885_MPT_CE-2027.pdf",
        "filename": "fpsc_mpt_ce_2027_public_notice.pdf",
        "subject": "General Knowledge", "topic": "CSS Screening Test (MPT)",
        "source_title": "MCQs Based Preliminary Test (MPT), CSS CE-2027 Public Notice",
    },
    {
        "url": "https://na.gov.pk/uploads/documents/63e602ca09b5d_461.pdf",
        "filename": "constitution_of_pakistan_1973_amended_2018.pdf",
        "subject": "Governance and Constitutional Studies", "topic": "Constitutional Framework",
        "source_title": "Constitution of the Islamic Republic of Pakistan, 1973 (as modified up to 31 May 2018)",
    },
    {
        "url": "https://finance.gov.pk/survey/chapter_26/Overview%20of%20the%20economy_26.pdf",
        "filename": "economic_survey_2025_26_overview.pdf",
        "subject": "Current Affairs", "topic": "Economy",
        "source_title": "Pakistan Economic Survey 2025-26 - Overview of the Economy",
    },
    {
        "url": "https://finance.gov.pk/survey/chapter_26/17_Climate_Change.pdf",
        "filename": "economic_survey_2025_26_climate_change.pdf",
        "subject": "Current Affairs", "topic": "Environment and Climate",
        "source_title": "Pakistan Economic Survey 2025-26 - Chapter 17: Climate Change",
    },
    {
        "url": "https://finance.gov.pk/survey/chapter_26/8_Trade%20and%20Payments.pdf",
        "filename": "economic_survey_2025_26_trade_and_payments.pdf",
        "subject": "International Relations", "topic": "Trade and External Sector",
        "source_title": "Pakistan Economic Survey 2025-26 - Chapter 8: Trade and Payments",
    },
    {
        "url": "http://web.archive.org/web/20251116081212/https://www.fpsc.gov.pk/assets/media/2024-08-23-11-18-12-Final_Approved_Rules_CE-2019_and_onwards.pdf",
        "filename": "fpsc_css_ce_rules_2019_archived.pdf",
        "subject": "General Knowledge", "topic": "CSS Exam Rules (Written, Medical, Psychological Assessment, Viva Voce)",
        "source_title": "CSS Competitive Examination Rules, 2019 (FPSC; archived copy -- live fpsc.gov.pk URL 404s)",
    },
    {
        "url": "http://web.archive.org/web/20251216062719/https://www.fpsc.gov.pk/assets/media/2025-10-16-06-52-10-1.-Instructions-to-the-Candidates-for-Psychological-Assessment-%26-Viva-Voce.pdf",
        "filename": "fpsc_viva_voce_instructions_ce2025_archived.pdf",
        "subject": "General Knowledge", "topic": "CSS Viva Voce and Psychological Assessment Instructions",
        "source_title": "Instructions to the Candidates for Psychological Assessment & Viva Voce, CSS CE-2025 (FPSC; archived copy)",
    },
    {
        "url": "https://peacekeeping.un.org/sites/default/files/pakistan.pdf",
        "filename": "un_peacekeeping_pakistan_statement.pdf",
        "subject": "International Relations", "topic": "Pakistan and UN Peacekeeping",
        "source_title": "Pakistan's statement to the UN Special Committee on Peacekeeping Operations (peacekeeping.un.org)",
    },
    {
        "url": "https://www.pips.gov.pk/wp-content/uploads/2020/09/Constitution_18th_amendment_Act2010and_Devolution.pdf",
        "filename": "constitution_18th_amendment_act_2010_devolution.pdf",
        "subject": "Governance and Constitutional Studies", "topic": "18th Constitutional Amendment and Devolution",
        "source_title": "Constitution (18th Amendment) Act 2010 and Devolution (Pakistan Institute for Parliamentary Services)",
    },
]

chunks_out = BASE_DIR / "07_Processed_Chunks" / "pdf_chunks.jsonl"
raw_dir = BASE_DIR / "01_Raw_Data"

for job in PDF_JOBS:
    local_path = raw_dir / job["filename"]
    print("Processing:", job["source_title"])
    try:
        pre_existing = local_path.exists()
        if not pre_existing:
            download_or_explain(job["url"], local_path, job["source_title"])
        pages = extract_pdf_text(local_path)
        if not pages:
            raise ValueError("no extractable text (likely a scanned/image PDF -- needs OCR, out of scope here)")
        chunk_count = 0
        for page in pages:
            records = make_chunk_records(
                page["text"], subject=job["subject"], topic=job["topic"], source_type="official_document",
                source_title=job["source_title"], source_url=job["url"], page_number=page["page_number"],
            )
            append_jsonl(records, chunks_out)
            chunk_count += len(records)
        PROCESSING_LOG["success"].append({
            "source": job["source_title"], "url": job["url"],
            "method": "pre-existing local file" if pre_existing else "downloaded",
            "pages": len(pages), "chunks": chunk_count,
        })
        print("  OK:", len(pages), "page(s),", chunk_count, "chunk(s)")
    except Exception as exc:
        PROCESSING_LOG["failed"].append({"source": job["source_title"], "url": job["url"], "error": str(exc)})
        print("  FAILED:", exc)


Processing: CSS Competitive Examination (CE-2027) Public Notice
  OK: 1 page(s), 2 chunk(s)
Processing: MCQs Based Preliminary Test (MPT), CSS CE-2027 Public Notice
  OK: 1 page(s), 2 chunk(s)
Processing: Constitution of the Islamic Republic of Pakistan, 1973 (as modified up to 31 May 2018)


  OK: 225 page(s), 312 chunk(s)
Processing: Pakistan Economic Survey 2025-26 - Overview of the Economy
  OK: 15 page(s), 30 chunk(s)
Processing: Pakistan Economic Survey 2025-26 - Chapter 17: Climate Change


  OK: 24 page(s), 46 chunk(s)
Processing: Pakistan Economic Survey 2025-26 - Chapter 8: Trade and Payments


  OK: 32 page(s), 56 chunk(s)
Processing: CSS Competitive Examination Rules, 2019 (FPSC; archived copy -- live fpsc.gov.pk URL 404s)
  OK: 22 page(s), 40 chunk(s)
Processing: Instructions to the Candidates for Psychological Assessment & Viva Voce, CSS CE-2025 (FPSC; archived copy)
  OK: 2 page(s), 4 chunk(s)
Processing: Pakistan's statement to the UN Special Committee on Peacekeeping Operations (peacekeeping.un.org)
  OK: 2 page(s), 3 chunk(s)
Processing: Constitution (18th Amendment) Act 2010 and Devolution (Pakistan Institute for Parliamentary Services)


  OK: 212 page(s), 247 chunk(s)


## 7b. Process the official FPSC syllabus (all compulsory + optional subjects)

A single 158-page FPSC document -- "Revised Syllabi for CSS Competitive Examination" -- covers the
scheme of the exam (marks per paper), the full optional-subjects group list, and a detailed topic-by-
topic syllabus for every compulsory and optional subject (English Essay, Precis, General Science &
Ability, Current Affairs, Pakistan Affairs, Islamic Studies, Political Science, Public Administration,
International Relations, International Law, Economics, Sociology, History, Criminology, Psychology,
Philosophy, and more).

**Provenance, honestly stated:** this exact file is no longer reachable at a live `fpsc.gov.pk` URL (the
site has since migrated and old `/assets/media/...` paths now 404). What's used below is the Internet
Archive's Wayback Machine snapshot of it, captured 2024-11-04, of a document FPSC itself published and
dated "Updated on: 7th July, 2015" for "CE-2016 and onwards". Independent reporting on the CSS 2026/2027
cycles says the compulsory-subject structure and marks have not changed since, but **verify this
against FPSC's current advertisement before treating any specific topic/mark as still current** --
don't take this notebook's word for it.

Because one file covers ~52 subjects by page range, this cell detects each subject's own
`PAPER: <NAME> (<N> MARKS)` heading and tags chunks with the subject it actually found on that page
(carried forward until the next heading), rather than one static subject/topic for the whole file.
**Fixed and re-verified:** an earlier version of this regex required each word of the subject name to
start with a letter, which silently broke on names containing `&` or a parenthetical as a separate word
("ENGLISH (PRECIS & COMPOSITION)", "GENERAL SCIENCE & ABILITY") -- those pages matched nothing and
inherited the previous heading. The current regex instead captures everything between `PAPER:` and the
first `(<N> MARKS)` non-greedily (with `re.DOTALL`, since the heading can span lines) and is verified
locally to detect 55 of the ~52 listed subjects correctly (it fires twice for a few multi-page intros
like Islamic Studies/Pashto/Persian, which is fine -- same label both times). Check `issues` in section
11 and the printed detection count below if you still see a subject missing after adding more sources.

**Promoted from Task 8 / Experiment C (retrieval-quality investigation):** every page of this document
repeats a running header, `"<page#> Revised Scheme and Syllabus for CSS Competitive Examination"`. 9
pages are *only* that header with nothing else, and it's a *prefix* on 155 of the other 161 chunks --
measured to create near-duplicate, subject-agnostic embeddings that out-ranked genuinely relevant
content for broad queries (e.g. the real CSS-exam-structure answer ranked #17, behind header-only pages
from unrelated subjects like Arabic/Balochi/Urdu, which scored higher purely on shared boilerplate
wording). Stripping it here, at extraction, before chunking -- rather than as a separate post-processing
pass -- means the 9 header-only pages naturally produce zero chunks (their text becomes empty after
stripping) and the other 155 are chunked clean from the start. Measured effect on the 20-question
evaluation (section 13): mean Precision@5 0.510 -> 0.640, zero regressions. The regex is anchored to the
start of the page (`^`), so it only ever removes this exact running header -- a genuine reference to
"the CSS Competitive Examination" inside a sentence, anywhere other than this literal repeated banner,
is untouched.

In [12]:
import re as _re

SYLLABUS_HEADER_RE = _re.compile(
    r"PAPER\s*:\s*(.+?)\(\s*\d+\s*MARKS?\s*\)",
    _re.IGNORECASE | _re.DOTALL,
)

RUNNING_HEADER_RE = _re.compile(
    r"^\s*\d+\s*Revised Scheme and Syllabus for CSS Competitive Examination\s*",
    _re.IGNORECASE,
)


def extract_official_syllabus_sections(pdf_path: Path, scheme_start_page: int = 4) -> list[dict]:
    """Returns [{"page_number", "subject_label", "text"}], carrying the last detected PAPER: heading
    forward across continuation pages. Pages before the first detected heading (the scheme/marks tables)
    are labelled "CSS Exam Scheme and Marks". The repeated running header is stripped from each page's
    text before it's returned (and subject detection runs on the original, unstripped text first, so
    stripping never affects which subject a page is tagged with) -- a page that is only that header
    becomes empty after stripping and is skipped entirely, producing no chunk."""
    doc = fitz.open(pdf_path)
    sections = []
    current_label = "CSS Exam Scheme and Marks"
    for i, page in enumerate(doc, start=1):
        text = page.get_text("text")
        if i < scheme_start_page or not text.strip():
            continue
        match = SYLLABUS_HEADER_RE.search(text)
        if match:
            current_label = _re.sub(r"\s+", " ", match.group(1)).strip().title()
        cleaned_text = RUNNING_HEADER_RE.sub("", text)
        if not cleaned_text.strip():
            continue
        sections.append({"page_number": i, "subject_label": current_label, "text": cleaned_text})
    doc.close()
    return sections


SYLLABUS_JOB = {
    "url": "http://web.archive.org/web/20241104213215/https://www.fpsc.gov.pk/assets/media/2024-10-22-01-45-26-Syllabus_for_CE-2016_and_onwards.pdf",
    "filename": "fpsc_official_css_syllabus_archived_2024-11-04.pdf",
    "source_title": "Revised Syllabi for CSS Competitive Examination (FPSC; archived 2024-11-04; dated for CE-2016 onwards)",
    "topic": "Official CSS Syllabus (Scheme, Marks and Subject Topics)",
}

local_path = BASE_DIR / "01_Raw_Data" / SYLLABUS_JOB["filename"]
chunks_out = BASE_DIR / "07_Processed_Chunks" / "pdf_chunks.jsonl"
print("Processing:", SYLLABUS_JOB["source_title"])
try:
    pre_existing = local_path.exists()
    if not pre_existing:
        download_or_explain(SYLLABUS_JOB["url"], local_path, SYLLABUS_JOB["source_title"])
    sections = extract_official_syllabus_sections(local_path)
    if not sections:
        raise ValueError("no extractable text")
    detected_subjects = {s["subject_label"] for s in sections} - {"CSS Exam Scheme and Marks"}
    chunk_count = 0
    for section in sections:
        records = make_chunk_records(
            section["text"], subject=section["subject_label"], topic=SYLLABUS_JOB["topic"],
            source_type="official_document", source_title=SYLLABUS_JOB["source_title"],
            source_url=SYLLABUS_JOB["url"], page_number=section["page_number"],
        )
        append_jsonl(records, chunks_out)
        chunk_count += len(records)
    PROCESSING_LOG["success"].append({
        "source": SYLLABUS_JOB["source_title"], "url": SYLLABUS_JOB["url"],
        "method": "pre-existing local file" if pre_existing else "downloaded (Wayback Machine snapshot)",
        "pages": len(sections), "chunks": chunk_count,
    })
    print(f"  OK: {len(sections)} page(s), {chunk_count} chunk(s), {len(detected_subjects)} distinct subject headings detected")
    print("  subjects found:", sorted(detected_subjects))
except Exception as exc:
    PROCESSING_LOG["failed"].append({"source": SYLLABUS_JOB["source_title"], "url": SYLLABUS_JOB["url"], "error": str(exc)})
    print("  FAILED:", exc)


Processing: Revised Syllabi for CSS Competitive Examination (FPSC; archived 2024-11-04; dated for CE-2016 onwards)


  OK: 151 page(s), 154 chunk(s), 52 distinct subject headings detected
  subjects found: ['Accountancy & Auditing', 'Agriculture & Forestry', 'Anthropology', 'Applied Mathematics', 'Arabic', 'Balochi', 'Botany', 'British History', 'Business Administration', 'Chemistry', 'Comparative Study Of Major Religions', 'Computer Science', 'Constitutional Law', 'Criminology', 'Current Affairs', 'Economics', 'English (Precis & Composition)', 'English Essay', 'English Literature', 'Environmental Science', 'European History', 'Gender Studies', 'General Science & Ability', 'Geography', 'Geology', 'Governance & Public Policies', 'History Of Pakistan & India', 'History Of Usa', 'International Law', 'International Relations', 'Islamic History & Culture', 'Islamic Studies', 'Journalism & Mass Communication', 'Law', 'Mercantile Law', 'Muslim Law & Jurisprudence', 'Pakistan Affairs', 'Pashto', 'Persian', 'Philosophy', 'Physics', 'Political Science', 'Psychology', 'Public Administration', 'Punjabi', 'Pure M

## 8. Process web articles → chunks

Four web sources verified so far: MOFA's foreign-policy page, the UN Charter's official full text, the
UN's own "Structure" page (principal organs: Security Council, General Assembly, ECOSOC, ICJ,
Secretariat), and a MOFA press release on Pakistan's UN peacekeeping role -- added specifically to close
the "role of the UN" coverage gap the retrieval evaluation (section 13) first surfaced. Several other
official portals (FPSC, PBS, SBP, WTO) were checked and rejected here because they are client-rendered
or table/nav-heavy and don't yield real article prose from a plain fetch — see `sources.json`'s
`portal_references_not_auto_scraped` and `excluded_or_failed` lists before re-adding them. Only use
sources you have the right to scrape, and respect `robots.txt`.

In [13]:
WEB_JOBS = [
    {
        "url": "https://mofa.gov.pk/foreign-policy",
        "subject": "International Relations", "topic": "Pakistan's Foreign Policy Principles",
    },
    {
        "url": "https://www.un.org/en/about-us/un-charter/full-text",
        "subject": "International Relations", "topic": "United Nations Charter (Full Text)",
    },
    {
        "url": "https://www.un.org/en/node/44718",
        "subject": "International Relations", "topic": "United Nations Principal Organs",
    },
    {
        "url": "https://mofa.gov.pk/press-releases/pakistani-peacekeepers-continue-to-exemplify-countrys-steadfast-commitment-to-world-peace",
        "subject": "International Relations", "topic": "Pakistan and UN Peacekeeping",
    },
]

chunks_out = BASE_DIR / "07_Processed_Chunks" / "web_chunks.jsonl"
for job in WEB_JOBS:
    print("Processing:", job["url"])
    try:
        article = with_retries(extract_web_article, job["url"], retries=3, backoff_seconds=3.0)
        if not article:
            raise ValueError("extraction returned no content (page may be JS-rendered or nav-only)")
        records = make_chunk_records(
            article["text"], subject=job["subject"], topic=job["topic"], source_type="official_document",
            source_title=article["title"], source_url=job["url"],
        )
        append_jsonl(records, chunks_out)
        PROCESSING_LOG["success"].append({"source": article["title"], "url": job["url"], "method": "scraped", "chunks": len(records)})
        print("  OK:", article["title"], "->", len(records), "chunk(s)")
    except Exception as exc:
        PROCESSING_LOG["failed"].append({"source": job["url"], "url": job["url"], "error": str(exc)})
        print("  FAILED:", exc)


Processing: https://mofa.gov.pk/foreign-policy


  OK: Foreign Policy -> 2 chunk(s)
Processing: https://www.un.org/en/about-us/un-charter/full-text


  OK: United Nations Charter (full text) | United Nations -> 33 chunk(s)
Processing: https://www.un.org/en/node/44718


  OK: UN Structure | United Nations -> 2 chunk(s)
Processing: https://mofa.gov.pk/press-releases/pakistani-peacekeepers-continue-to-exemplify-countrys-steadfast-commitment-to-world-peace


  OK: Pakistani peacekeepers continue to exemplify country’s steadfast commitment to world peace -> 1 chunk(s)


## 9. Process YouTube transcripts → chunks

Three real, verified CSS mock-interview videos from a CSS coaching academy's channel (confirmed to exist
via YouTube's oembed endpoint, and confirmed to have fetchable captions). These are coaching-academy
mock interviews, not official FPSC interview recordings — `source_type` stays `youtube_transcript` and
nothing here should be presented as an official transcript.

In [14]:
YOUTUBE_JOBS = [
    {
        "video_id": "CS47XiOlxT4", "subject": "General Knowledge", "topic": "CSS Viva Voce / Interview Preparation",
        "title": "Fizzah Mansur (FSP) 44th Position in CSS-2024 | Mock Interview by CSPs Academy Islamabad",
    },
    {
        "video_id": "SQszUBeWmRo", "subject": "General Knowledge", "topic": "CSS Viva Voce / Interview Preparation",
        "title": "Hamza Munir (PAS) 24th Position in CSS-2025 | Mock Interview by CSPs Academy Islamabad",
    },
    {
        "video_id": "-Jy6W5kXSS0", "subject": "General Knowledge", "topic": "CSS Viva Voce / Interview Preparation",
        "title": "Mahnoor Mirwani (PAS) 10th Position in CSS-2025 | Mock Interview by CSPs Academy Islamabad",
    },
]

chunks_out = BASE_DIR / "07_Processed_Chunks" / "youtube_chunks.jsonl"
for job in YOUTUBE_JOBS:
    print("Processing:", job["title"])
    try:
        text = with_retries(fetch_youtube_transcript, job["video_id"], retries=2, backoff_seconds=3.0)
        if not text:
            raise ValueError("no captions available for this video")
        url = f"https://www.youtube.com/watch?v={job['video_id']}"
        records = make_chunk_records(
            text, subject=job["subject"], topic=job["topic"], source_type="youtube_transcript",
            source_title=job["title"], source_url=url,
        )
        append_jsonl(records, chunks_out)
        PROCESSING_LOG["success"].append({"source": job["title"], "url": url, "method": "youtube transcript", "chunks": len(records)})
        print("  OK:", len(records), "chunk(s)")
    except Exception as exc:
        PROCESSING_LOG["failed"].append({"source": job["title"], "url": job["video_id"], "error": str(exc)})
        print("  FAILED:", exc)


Processing: Fizzah Mansur (FSP) 44th Position in CSS-2024 | Mock Interview by CSPs Academy Islamabad


  OK: 16 chunk(s)
Processing: Hamza Munir (PAS) 24th Position in CSS-2025 | Mock Interview by CSPs Academy Islamabad


  OK: 13 chunk(s)
Processing: Mahnoor Mirwani (PAS) 10th Position in CSS-2025 | Mock Interview by CSPs Academy Islamabad


  OK: 9 chunk(s)


## 9b. Source processing summary

In [15]:
print("=== Source processing summary (provenance) ===")
print("Succeeded:", len(PROCESSING_LOG["success"]))
for s in PROCESSING_LOG["success"]:
    method, source, url, n_chunks = s["method"], s["source"], s["url"], s.get("chunks", "?")
    print(f"  OK   [{method}] {source} ({n_chunks} chunk(s)) -- {url}")
print("Failed:", len(PROCESSING_LOG["failed"]))
for f in PROCESSING_LOG["failed"]:
    print("  FAIL", f["source"], "-", f["error"])


=== Source processing summary (provenance) ===
Succeeded: 18
  OK   [pre-existing local file] CSS Competitive Examination (CE-2027) Public Notice (2 chunk(s)) -- https://www.fpsc.gov.pk/uploads/content/1794127244067_CSS_2027_Advertisement.pdf
  OK   [pre-existing local file] MCQs Based Preliminary Test (MPT), CSS CE-2027 Public Notice (2 chunk(s)) -- https://www.fpsc.gov.pk/uploads/content/1785753970885_MPT_CE-2027.pdf
  OK   [pre-existing local file] Constitution of the Islamic Republic of Pakistan, 1973 (as modified up to 31 May 2018) (312 chunk(s)) -- https://na.gov.pk/uploads/documents/63e602ca09b5d_461.pdf
  OK   [pre-existing local file] Pakistan Economic Survey 2025-26 - Overview of the Economy (30 chunk(s)) -- https://finance.gov.pk/survey/chapter_26/Overview%20of%20the%20economy_26.pdf
  OK   [pre-existing local file] Pakistan Economic Survey 2025-26 - Chapter 17: Climate Change (46 chunk(s)) -- https://finance.gov.pk/survey/chapter_26/17_Climate_Change.pdf
  OK   [pre-existin

## 10. Load, deduplicate and sanity-check chunks (before spending any embedding compute)

Loads every `*_chunks.jsonl` file from `07_Processed_Chunks/`. Because sections 7-9 append (so
re-running one while iterating doesn't wipe the others sharing a file), **re-running a section twice
duplicates its chunks in the file** -- harmless now that `chunk_id` is a stable hash of (source, page,
position): the loader below deduplicates by `chunk_id`, so a chunk re-added by a re-run collapses back
into one. This is a safe, demonstrable removal (identical id = identical source+page+position+content)
-- it never touches the jsonl files on disk, and it never drops a short-but-real chunk just for being
short; it only drops exact re-occurrences of something already kept.

In [16]:
all_records_raw = []
for path in sorted((BASE_DIR / "07_Processed_Chunks").glob("*_chunks.jsonl")):
    with open(path, encoding="utf-8") as f:
        all_records_raw.extend(json.loads(line) for line in f if line.strip())

seen_ids = set()
all_records = []
reoccurrences = 0
for r in all_records_raw:
    if r["chunk_id"] in seen_ids:
        reoccurrences += 1
        continue
    seen_ids.add(r["chunk_id"])
    all_records.append(r)

short_chunks = [r for r in all_records if len(r["content"].split()) < 20]

print(f"Loaded {len(all_records_raw)} raw row(s) from disk.")
print(f"Dropped {reoccurrences} re-occurrence(s) of an already-seen chunk_id (safe: identical content).")
print(f"Kept {len(all_records)} distinct chunk(s) to embed.")
print(f"FLAGGED (not removed): {len(short_chunks)} chunk(s) under 20 words -- inspect before trusting "
      f"them as retrieval evidence; most are divider/continuation pages, not broken extraction.")
if not all_records:
    print("No chunks found yet -- run sections 7-9 first.")


Loaded 1944 raw row(s) from disk.
Dropped 972 re-occurrence(s) of an already-seen chunk_id (safe: identical content).
Kept 972 distinct chunk(s) to embed.
FLAGGED (not removed): 82 chunk(s) under 20 words -- inspect before trusting them as retrieval evidence; most are divider/continuation pages, not broken extraction.


## 11. Build embeddings + FAISS index

Embeds each distinct chunk's `content` with `all-MiniLM-L6-v2` (small, CPU-friendly, ~80MB — fine on
Colab, not meant to run on the laptop backend) in batches, normalizes the vectors (so inner product =
cosine similarity), builds the FAISS index, and saves the index, its metadata, AND a separate
`index_config.json` recording exactly what was used to build it -- model name, embedding dimension,
index type, chunk count, and a content hash -- so a reload later (section 11b, and the "fresh runtime"
step in the README's execution checklist) can verify it's reading back the same thing that was built
here, not just *some* index.

In [17]:
import hashlib as _hashlib
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

EMBED_BATCH_SIZE = 32  # generate in batches rather than one giant encode() call

model = SentenceTransformer(EMBEDDING_MODEL_NAME)

if all_records:
    texts = [r["content"] for r in all_records]
    embeddings = model.encode(texts, batch_size=EMBED_BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True)
    embeddings = np.asarray(embeddings, dtype="float32")
    embedding_dim = embeddings.shape[1]
    assert embeddings.shape[0] == len(all_records), "embedding count must match chunk count"

    index = faiss.IndexFlatIP(embedding_dim)  # inner product on normalized vectors = cosine similarity
    index.add(embeddings)
    assert index.ntotal == len(all_records), "every chunk must have exactly one vector in the index"

    embed_dir = BASE_DIR / "08_Embeddings"
    faiss.write_index(index, str(embed_dir / "css_knowledge_base.faiss"))
    with open(embed_dir / "css_knowledge_base_metadata.jsonl", "w", encoding="utf-8") as f:
        for r in all_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    content_fingerprint = _hashlib.sha1("".join(sorted(seen_ids)).encode()).hexdigest()[:16]
    index_config = {
        "model_name": EMBEDDING_MODEL_NAME,
        "embedding_dim": embedding_dim,
        "index_type": "faiss.IndexFlatIP (cosine via normalized inner product)",
        "normalize_embeddings": True,
        "embed_batch_size": EMBED_BATCH_SIZE,
        "chunk_count": len(all_records),
        "chunk_id_set_fingerprint": content_fingerprint,
        "built_at_utc": __import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat(),
    }
    with open(embed_dir / "index_config.json", "w", encoding="utf-8") as f:
        json.dump(index_config, f, indent=2)

    print("Saved index, metadata and index_config.json to", embed_dir)
    print("index_config:", index_config)
else:
    print("Skipped -- no chunks loaded in section 10.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Saved index, metadata and index_config.json to C:\Users\kinza\OneDrive\Desktop\css-mock-interview-coach\research\colab\output_promoted_baseline\InterviewIQ_Dataset\08_Embeddings
index_config: {'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_dim': 384, 'index_type': 'faiss.IndexFlatIP (cosine via normalized inner product)', 'normalize_embeddings': True, 'embed_batch_size': 32, 'chunk_count': 972, 'chunk_id_set_fingerprint': 'd686bd7fdc27dc6e', 'built_at_utc': '2026-11-17T21:31:54.716684+00:00'}


## 11b. Immediate reload check (same runtime)

Reads the index straight back from disk and confirms its vector count and dimension match what section
11 just reported, then runs one throwaway query -- a cheap, same-session sanity check. **This is not**
the real persistence test: that specifically requires a fresh Colab runtime (Runtime -> Disconnect and
delete runtime, then reconnect and re-mount Drive) so you're reading back something this process never
held in memory -- see the execution checklist in the README for that step, which this cell does not
replace.

In [18]:
if all_records:
    reloaded_index = faiss.read_index(str(embed_dir / "css_knowledge_base.faiss"))
    with open(embed_dir / "index_config.json", encoding="utf-8") as f:
        reloaded_config = json.load(f)

    assert reloaded_index.ntotal == reloaded_config["chunk_count"], "reloaded vector count must match index_config"
    assert reloaded_index.d == reloaded_config["embedding_dim"], "reloaded dimension must match index_config"

    probe_vec = model.encode(["Pakistan CSS examination"], normalize_embeddings=True).astype("float32")
    scores, idxs = reloaded_index.search(probe_vec, 1)
    print(f"Reloaded index OK: {reloaded_index.ntotal} vectors, dim={reloaded_index.d}.")
    print(f"Throwaway probe query top-1 score: {scores[0][0]:.3f} (sanity check only, not a real eval query).")
else:
    print("Skipped -- nothing was built in section 11.")


Reloaded index OK: 972 vectors, dim=384.
Throwaway probe query top-1 score: 0.692 (sanity check only, not a real eval query).


## 12. Dataset quality + subject distribution report

Checks every chunk for empty content, duplicate content, missing subject/topic/source attribution, and
suspiciously short chunks (flagged, never auto-removed -- a one-page subject like Philosophy or Law is
legitimately short; only exact re-occurrences get dropped, back in section 10). Then reports chunk count,
percentage of total, and source-document count per subject, and explicitly calls out any subject over
25% of the total as a possible retrieval-bias risk worth watching in section 13's results -- it does not
rebalance anything automatically; forcing equal counts across subjects of very different real size would
just hide the signal, not fix it.

In [19]:
from collections import Counter, defaultdict

issues = []
seen_content_hashes = set()

for r in all_records:
    chunk_id, source_title = r.get("chunk_id", "?"), r.get("source_title", "?")
    label = f"{chunk_id} ({source_title})"
    content = r.get("content", "")
    if not content or not content.strip():
        issues.append(label + ": empty content")
        continue
    word_count = len(content.split())
    if word_count < 20:
        issues.append(f"{label}: suspiciously short chunk ({word_count} words) -- flagged for review, not removed")
    content_hash = hash(content.strip().lower())
    if content_hash in seen_content_hashes:
        issues.append(label + ": duplicate content of another chunk (unexpected post-section-10 -- investigate)")
    seen_content_hashes.add(content_hash)
    for field_name in ("subject", "topic", "source_title", "source_type"):
        if not r.get(field_name):
            issues.append(f"{label}: missing \'{field_name}\'")
    if r.get("source_type") != "youtube_transcript" and not r.get("source_url"):
        issues.append(label + ": missing source_url (no attribution)")

print(f"Checked {len(all_records)} chunks: {len(issues)} issue(s) found")
for issue in issues[:50]:
    print("  -", issue)
if len(issues) > 50:
    print(f"  ... and {len(issues) - 50} more")

total = len(all_records)
by_subject = Counter(r["subject"] for r in all_records)
docs_by_subject = defaultdict(set)
for r in all_records:
    docs_by_subject[r["subject"]].add(r["source_title"])

print("\n=== Subject distribution ===")
for subject, count in by_subject.most_common():
    pct = 100 * count / total if total else 0
    flag = "  <-- over 25% of the index; watch for retrieval bias toward this subject" if pct > 25 else ""
    print(f"  {subject:45s} chunks={count:4d} ({pct:5.1f}%)  source_docs={len(docs_by_subject[subject])}{flag}")

thin = [s for s, c in by_subject.items() if c <= 2]
if thin:
    print(f"\nThin subjects (<=2 chunks, may under-retrieve against more common topics): {thin}")


Checked 972 chunks: 95 issue(s) found
  - a9ed93aafaeebd22 (Constitution of the Islamic Republic of Pakistan, 1973 (as modified up to 31 May 2018)): suspiciously short chunk (19 words) -- flagged for review, not removed
  - abf984b968ac90ea (Constitution (18th Amendment) Act 2010 and Devolution (Pakistan Institute for Parliamentary Services)): suspiciously short chunk (10 words) -- flagged for review, not removed
  - a87d91c2bba685f7 (Constitution (18th Amendment) Act 2010 and Devolution (Pakistan Institute for Parliamentary Services)): suspiciously short chunk (10 words) -- flagged for review, not removed
  - a87d91c2bba685f7 (Constitution (18th Amendment) Act 2010 and Devolution (Pakistan Institute for Parliamentary Services)): duplicate content of another chunk (unexpected post-section-10 -- investigate)
  - 837194de27f099f4 (Constitution (18th Amendment) Act 2010 and Devolution (Pakistan Institute for Parliamentary Services)): suspiciously short chunk (8 words) -- flagged for revie

## 13. Retrieval evaluation (real FAISS retrieval, run here -- do not hand-copy fake numbers)

Runs a broad set of CSS exam/interview questions covering every compulsory subject plus the optional
subjects and topics this knowledge base targets, retrieves the real top-5 FAISS matches for each, and
reports a subject-match-based Precision@5: for each question, `expected_subjects` lists which subject
label(s) would count as on-topic, and precision@5 = (retrieved chunks whose subject is in that list) / 5.
**This is a topical-relevance proxy, not a human judgment of whether the retrieved text actually answers
the question** -- read the printed excerpts yourself for that; the metric only tells you whether FAISS
is even looking in the right subject area.

In [20]:
def search(query: str, top_k: int = TOP_K_DEFAULT):
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        rec = all_records[idx]
        results.append({"score": float(score), **rec})
    return results


EVALUATION_QUESTIONS = [
    {"q": "What are the eligibility requirements for Pakistan\'s CSS examination?", "expected_subjects": ["General Knowledge"]},
    {"q": "Explain the structure of the CSS written examination.", "expected_subjects": ["General Knowledge", "CSS Exam Scheme and Marks"]},
    {"q": "What qualities are evaluated during the CSS viva voce?", "expected_subjects": ["General Knowledge"]},
    {"q": "What are the major constitutional developments in Pakistan?", "expected_subjects": ["Governance and Constitutional Studies", "Pakistan Affairs"]},
    {"q": "Explain the key provisions of the Constitution of Pakistan regarding fundamental rights.", "expected_subjects": ["Governance and Constitutional Studies"]},
    {"q": "Discuss the significance of the 18th Constitutional Amendment.", "expected_subjects": ["Governance and Constitutional Studies"]},
    {"q": "Explain Pakistan\'s federal system.", "expected_subjects": ["Governance and Constitutional Studies", "Political Science", "Governance & Public Policies", "Public Administration"]},
    {"q": "Explain the role and functions of the Federal Public Service Commission.", "expected_subjects": ["General Knowledge", "Governance and Constitutional Studies"]},
    {"q": "Discuss the ideology of Pakistan and the Pakistan Movement.", "expected_subjects": ["Pakistan Affairs"]},
    {"q": "Discuss Pakistan\'s economic challenges.", "expected_subjects": ["Current Affairs", "Economics", "Pakistan Affairs"]},
    {"q": "Explain the causes and effects of inflation in a developing economy.", "expected_subjects": ["Economics", "Current Affairs"]},
    {"q": "Explain the concept of Ijtihad in Islamic jurisprudence.", "expected_subjects": ["Islamic Studies"]},
    {"q": "What qualities make a strong CSS English Essay?", "expected_subjects": ["English Essay"]},
    {"q": "Explain the key steps in writing an effective precis.", "expected_subjects": ["English (Precis & Composition)"]},
    {"q": "Explain the basic structure of the solar system and the Earth\'s place in it.", "expected_subjects": ["General Science & Ability"]},
    {"q": "Explain the role of the United Nations in international relations.", "expected_subjects": ["International Relations"]},
    {"q": "What are the major foreign policy challenges facing Pakistan?", "expected_subjects": ["International Relations", "Pakistan Affairs"]},
    {"q": "Explain the sources of international law.", "expected_subjects": ["International Law"]},
    {"q": "Explain the theory of separation of powers.", "expected_subjects": ["Political Science"]},
    {"q": "Explain the key principles of public administration.", "expected_subjects": ["Public Administration"]},
]

eval_report = []
for item in EVALUATION_QUESTIONS:
    q, expected = item["q"], set(item["expected_subjects"])
    hits = search(q, top_k=5)
    subject_matches = sum(1 for r in hits if r["subject"] in expected)
    precision_at_5 = subject_matches / 5 if hits else 0.0
    source_diversity = len({r["source_title"] for r in hits})
    eval_report.append({
        "query": q, "expected_subjects": sorted(expected), "n_hits": len(hits),
        "precision_at_5_subject_match": precision_at_5, "source_diversity": source_diversity,
        "missing_evidence": subject_matches == 0,
        "top5": [{"score": r["score"], "subject": r["subject"], "source_title": r["source_title"],
                   "source_type": r["source_type"], "excerpt": r["content"][:160]} for r in hits],
    })

    print("=" * 90)
    print("Q:", q, "| expected subjects:", sorted(expected))
    if not hits:
        print("  (no results -- index is empty)")
        continue
    print(f"  precision@5 (subject-match proxy) = {precision_at_5:.2f} | source diversity = {source_diversity}/5")
    for r in hits:
        score, subj, title, excerpt = r["score"], r["subject"], r["source_title"][:55], r["content"][:120]
        print(f"  [{score:.3f}] subj={subj!r} | {title} :: {excerpt}...")

mean_precision = sum(e["precision_at_5_subject_match"] for e in eval_report) / len(eval_report)
zero_evidence = [e["query"] for e in eval_report if e["missing_evidence"]]
print("\n" + "=" * 90)
print(f"Mean precision@5 (subject-match proxy) across {len(eval_report)} questions: {mean_precision:.2f}")
print(f"Questions with ZERO subject-matching evidence in top-5 ({len(zero_evidence)}):")
for q in zero_evidence:
    print("  -", q)

with open(BASE_DIR / "09_Evaluation_Dataset" / "retrieval_evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(eval_report, f, indent=2, ensure_ascii=False)
print("\nSaved full per-question report to 09_Evaluation_Dataset/retrieval_evaluation_report.json")


Q: What are the eligibility requirements for Pakistan's CSS examination? | expected subjects: ['General Knowledge']
  precision@5 (subject-match proxy) = 0.80 | source diversity = 4/5
  [0.724] subj='General Knowledge' | CSS Competitive Examination (CE-2027) Public Notice :: [[ No. F.2/1/2026-CE FEDERAL PUBLIC SERVICE COMMISSION Aga Khan Road Sector F-5/1 E-mail: fpsc@fpsc.gov.pk Website: www....
  [0.721] subj='General Knowledge' | MCQs Based Preliminary Test (MPT), CSS CE-2027 Public N :: [[ FEDERAL PUBLIC SERVICE COMMISSION Aga Khan Road Sector F-5/1 E-mail: fpsc@fpsc.gov.pk Website: www.fpsc.gov.pk No.F.2...
  [0.712] subj='CSS Exam Scheme and Marks' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: FEDERAL PUBLIC SERVICE COMMISSION REVISED SCHEME OF CSS COMPETITIVE EXAMINATION MAIN EXAMINATION (1200 MARKS) A: COMPULS...
  [0.665] subj='General Knowledge' | CSS Competitive Examination Rules, 2019 (FPSC; archived :: 16 7. Optional subjects (600 marks) The minimum standard

Q: What are the major constitutional developments in Pakistan? | expected subjects: ['Governance and Constitutional Studies', 'Pakistan Affairs']
  precision@5 (subject-match proxy) = 1.00 | source diversity = 3/5
  [0.667] subj='Governance and Constitutional Studies' | Constitution (18th Amendment) Act 2010 and Devolution ( :: th 18 Constitutional Amendment: A MILESTONE IN PAKISTAN'S PARLIAMENTARY DEVELOPMENT Honourable Dr Fehmida Mirza, former ...
  [0.664] subj='Pakistan Affairs' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: XXVI. Economic Conditions of Pakistan, the Most Recent Economic Survey, the Previous and Current Budgets, and the Proble...
  [0.661] subj='Governance and Constitutional Studies' | Constitution of the Islamic Republic of Pakistan, 1973  :: THE CONSTITUTION OF THE ISLAMIC REPUBLIC OF PAKISTAN [As modified upto the 31st May, 2018] NATIONAL ASSEMBLY OF PAKISTAN...
  [0.659] subj='Governance and Constitutional Studies' | Constitution of the Islamic Re

Q: Discuss the ideology of Pakistan and the Pakistan Movement. | expected subjects: ['Pakistan Affairs']
  precision@5 (subject-match proxy) = 0.40 | source diversity = 2/5
  [0.715] subj='History Of Pakistan & India' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: SUGGESTED READINGS S.No. TITLE AUTHOR 1. POLITICAL PARTIES IN PAKISTAN: 1947-1958 AFZAL, M. RAFIQUE, 1986 2. GOVERNMENT ...
  [0.703] subj='Political Science' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: S. No. Title Author 22. 21st Century Political Science: A Reference Handbook John T.Ishiyama, Marijke Breuning, 23. Poli...
  [0.700] subj='Pakistan Affairs' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: PAPER: PAKISTAN AFFAIRS (100 MARKS) I. Ideology of Pakistan-----definition and elucidation, historical aspects: Muslim r...
  [0.696] subj='Pakistan Affairs' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: XXVI. Economic Conditions of Pakistan, the Most Recent Economic

Q: Explain the concept of Ijtihad in Islamic jurisprudence. | expected subjects: ['Islamic Studies']
  precision@5 (subject-match proxy) = 0.40 | source diversity = 2/5
  [0.683] subj='Muslim Law & Jurisprudence' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: SUGGESTED READINGS S.No. Title Author 1. An Introduction to the Study of Islamic Law Dr. Hussain Hamid Hassan 2. Princip...
  [0.612] subj='Muslim Law & Jurisprudence' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: PAPER: MUSLIM LAW & JURISPRUDENCE (100 MARKS) I. The Sources of Islamic Law  The Quran  The Sunnah  Ijma  Qiyas  Ij...
  [0.546] subj='Islamic Studies' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: REQUIRED READINGS Sr. No. Title Author 1. Introduction to Islam Dr. Hamidullah 2. Islam: its meaning and Message Khurshi...
  [0.537] subj='Islamic Studies' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: 23. Studies in Muslim Political Thoughts and Administration S

Q: Explain the basic structure of the solar system and the Earth's place in it. | expected subjects: ['General Science & Ability']
  precision@5 (subject-match proxy) = 0.20 | source diversity = 1/5
  [0.365] subj='General Science & Ability' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: PAPER: GENERAL SCIENCE & ABILITY (100 MARKS) Part-I (General Science) 60 Marks I. Physical Sciences  Constituents and S...
  [0.311] subj='Geography' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: PAPER: GEOGRAPHY (100 MARKS) Part –I (Marks 50) Physical Geography I. Factors controlling landform development; endogeni...
  [0.245] subj='Geology' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: PAPER: GEOLOGY (100 MARKS) Part- I (50-marks) I. Introduction to Physical Geology Introduction and scope of geology, its...
  [0.216] subj='Physics' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: PAPER: PHYSICS (200 MARKS) PAPER—I (Marks-100) I. Mechanics  Vec


Q: Explain the sources of international law. | expected subjects: ['International Law']
  precision@5 (subject-match proxy) = 0.40 | source diversity = 2/5
  [0.716] subj='International Law' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: PAPER: INTERNATIONAL LAW (100 MARKS) I. Nature, Origin and Basis of International Law.  The Emergence of International ...
  [0.611] subj='International Law' | Revised Syllabi for CSS Competitive Examination (FPSC;  :: SUGGESTED READINGS S.No. Title Author 1. International Law Malcolm N. Shaw 2. Principles of Public International Law Lan...
  [0.473] subj='International Relations' | United Nations Charter (full text) | United Nations :: against abuses; - to develop self-government, to take due account of the political aspirations of the peoples, and to as...
  [0.453] subj='International Relations' | United Nations Charter (full text) | United Nations :: for the convening of meetings on the request of a majority of its members. Article 

## 14. Export final dataset

Copies the metadata JSONL (and a CSV view) into `10_Final_Export/` for download or for later review
before any backend integration decision.

In [21]:
import csv

export_dir = BASE_DIR / "10_Final_Export"
metadata_path = BASE_DIR / "08_Embeddings" / "css_knowledge_base_metadata.jsonl"

if metadata_path.exists():
    import shutil
    shutil.copy(metadata_path, export_dir / metadata_path.name)

    with open(metadata_path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    if rows:
        csv_path = export_dir / "css_knowledge_base_metadata.csv"
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        print("Exported:", csv_path)
    print("Exported:", export_dir / metadata_path.name)
else:
    print("Nothing to export yet — build the index in section 11 first.")


Exported: C:\Users\kinza\OneDrive\Desktop\css-mock-interview-coach\research\colab\output_promoted_baseline\InterviewIQ_Dataset\10_Final_Export\css_knowledge_base_metadata.csv
Exported: C:\Users\kinza\OneDrive\Desktop\css-mock-interview-coach\research\colab\output_promoted_baseline\InterviewIQ_Dataset\10_Final_Export\css_knowledge_base_metadata.jsonl


## Next steps

1. Add more verified sources to `PDF_JOBS` / `WEB_JOBS` / `YOUTUBE_JOBS` (sections 7-9) -- check each URL
   actually resolves and extracts real text before adding it (see `sources.json` for the method used
   here: HTTP status + sampled extracted text, not just a search-result snippet).
2. Re-run sections 10-14 whenever you add new sources: load/dedupe (10), rebuild the index (11), reload
   check (11b), re-check quality/subject distribution (12), re-run retrieval evaluation (13), re-export
   (14).
3. For the app's actual *question bank* (what candidates get asked), use
   `backend/scripts/generate_questions.py` in the main repo instead — it drafts questions in the schema
   `backend/app/rag/dataset.py` validates, and never auto-merges into the live dataset.
4. If you later want the FastAPI backend itself to query this FAISS index (instead of the current
   keyword retriever), that's a backend architecture change — raise it explicitly before installing
   `faiss-cpu` / `sentence-transformers` there, since the project's hardware rules currently rule out
   large local embedding models on the dev laptop.